In [1]:
# Install required libraries
!pip install tqdm requests networkx pandas

In [2]:

import requests
import pandas as pd
import networkx as nx
from tqdm import tqdm

In [5]:
# Define your topics and how many papers per topic
topics = {
    "Computer Science": "C41008148",
    "Genetics": "C54355233",
    "Biology": "C86803240",
    "Mathematics": "C33923547",
    "Environmental Science": "C39432304",
    "Physics": "C121332964",
    "Sociology": "C144133560"
}


# OpenAlex API base
BASE_URL = "https://api.openalex.org/works"

#get papers from all years


In [ ]:
#2000 per topic

# Helper to get works by concept (field of study)
def get_papers_by_topic(concept_id, concept_name, per_topic=2000):
    print(f"\nCollecting papers for: {concept_name}")
    papers = []
    cursor = "*"
    collected = 0

    while collected < per_topic:
        params = {
            "filter": f"concept.id:{concept_id},is_paratext:false,from_publication_date:2000-01-01,to_publication_date:2020-12-31",
            "per-page": 50,
            "cursor": cursor
        }
        response = requests.get(BASE_URL, params=params)
        if response.status_code != 200:
            print(f"⚠️ Failed to fetch data for {concept_name} (status code: {response.status_code})")
            break

        data = response.json()
        for result in data["results"]:
            if collected >= per_topic:
                break

            work_type = result.get("type", "")
            if work_type != "article":
                continue

            year = result.get("publication_year", None)
            if year is None or not (2000 <= year <= 2020):
                continue

            papers.append({
                "id": result["id"].split("/")[-1],
                "title": result["title"],
                "year": year,
                "concepts": [c["display_name"] for c in result.get("concepts", [])],
                "referenced_works": [r.split("/")[-1] for r in result.get("referenced_works", [])],
                "primary_topic": concept_name
            })
            collected += 1

        cursor = data["meta"].get("next_cursor")
        if cursor is None:
            break

    return papers


#get papers from each year

In [15]:
#per year 200 papers

def get_papers_by_topic(concept_id, concept_name, per_year=70):
    print(f"\nCollecting papers for: {concept_name}")
    papers = []

    # Loop through each year from 2000 to 2020
    for year in range(2000, 2021):
        print(f"  Year: {year}")

        year_collected = 0
        cursor = "*"

        while year_collected < per_year:
            params = {
                "filter": (
                    f"concept.id:{concept_id},"
                    f"is_paratext:false,"
                    f"publication_year:{year}"
                ),
                "per-page": 50,
                "cursor": cursor
            }

            response = requests.get(BASE_URL, params=params)
            if response.status_code != 200:
                print(f"    ⚠️ Failed to fetch data for {concept_name} in {year}")
                break

            data = response.json()
            results = data.get("results", [])

            # If no results left → stop early for this year
            if not results:
                break

            for result in results:
                if year_collected >= per_year:
                    break

                # Must be an article
                if result.get("type") != "article":
                    continue

                papers.append({
                    "id": result["id"].split("/")[-1],
                    "title": result["title"],
                    "year": year,
                    "concepts": [c["display_name"] for c in result.get("concepts", [])],
                    "referenced_works": [
                        r.split("/")[-1] for r in result.get("referenced_works", [])
                    ],
                    "primary_topic": concept_name
                })

                year_collected += 1

            # Move to next cursor page
            cursor = data["meta"].get("next_cursor")

            # No more results → stop year
            if cursor is None:
                break

        print(f"    → Collected {year_collected} papers for {year}")

    return papers


In [16]:
# Step 1: Download data
all_papers = []
for topic_name, concept_id in topics.items():
    all_papers.extend(get_papers_by_topic(concept_id, topic_name, 70))

print(f"\nTotal papers collected: {len(all_papers)}")





  Year: 2000
    → Collected 70 papers for 2000
  Year: 2001
    → Collected 70 papers for 2001
  Year: 2002
    → Collected 70 papers for 2002
  Year: 2003
    → Collected 70 papers for 2003
  Year: 2004
    → Collected 70 papers for 2004
  Year: 2005
    → Collected 70 papers for 2005
  Year: 2006
    → Collected 70 papers for 2006
  Year: 2007
    → Collected 70 papers for 2007
  Year: 2008
    → Collected 70 papers for 2008
  Year: 2009
    → Collected 70 papers for 2009
  Year: 2010
    → Collected 70 papers for 2010
  Year: 2011
    → Collected 70 papers for 2011
  Year: 2012
    → Collected 70 papers for 2012
  Year: 2013
    → Collected 70 papers for 2013
  Year: 2014
    → Collected 70 papers for 2014
  Year: 2015
    → Collected 70 papers for 2015
  Year: 2016
    → Collected 70 papers for 2016
  Year: 2017
    → Collected 70 papers for 2017
  Year: 2018
    → Collected 70 papers for 2018
  Year: 2019
    → Collected 70 papers for 2019
  Year: 2020
    → Collected 70 papers 

In [17]:
# Step 2: Create node dataframe
papers_df = pd.DataFrame(all_papers)
paper_ids = set(papers_df['id'])

# Step 3: Build citation edge list (only internal citations)
edges = []
for _, row in papers_df.iterrows():
    for ref in row["referenced_works"]:
        if ref in paper_ids:
            edges.append((row["id"], ref))

edges_df = pd.DataFrame(edges, columns=["cited", "cites"])
print(f"Total citation edges between papers: {len(edges_df)}")

# Step 4: Save results
papers_df.to_csv("papers_metadata.csv", index=False)
edges_df.to_csv("citation_edges.csv", index=False)




Total citation edges between papers: 37342


In [10]:
# Optional: Build and save a NetworkX graph
G = nx.DiGraph()
G.add_edges_from(edges)
nx.write_gml(G, "citation_graph.gml")

print("\n✅ Citation network saved: papers_metadata.csv, citation_edges.csv, citation_graph.gml")


✅ Citation network saved: papers_metadata.csv, citation_edges.csv, citation_graph.gml


In [11]:
# All paper IDs
all_ids = set(papers_df["id"])

# All nodes actually in the graph
graph_ids = set(G.nodes)

# Papers that are isolated (not involved in internal citation)
isolated_ids = all_ids - graph_ids

print(f"📄 Total papers: {len(all_ids)}")
print(f"🧠 Nodes in citation graph: {len(graph_ids)}")
print(f"❌ Isolated papers (not cited & don't cite internally): {len(isolated_ids)}")


📄 Total papers: 20731
🧠 Nodes in citation graph: 19457
❌ Isolated papers (not cited & don't cite internally): 1274
